***

# **About Automation**

***

This file is focused on automating the About pages on every excel file that we make for the indicators. The user needs to define the `sample_type`, `indicator_name`, dataframe that the possible years will be pulled from, and the date will be pulled automatically. The code also has the process of converting the `About Indicators.xlsx` file to a YAML type. From the YAML file, `dict_about.yaml`, more edits were made to include additional indicators. From this file, users can indicate the about section of a page by searching the file for the indicator name, and then editing the section dedicated to this indicator. 

The code works by linking to this YAML file, and extracting the information of the specific sample and indicator name the user wants. From there, this subset is converted to a pandas dataframe. This dataframe is parsed through to check for dynamic data (fields that change values regularly), such as year(s) or when the indicator was last updated. Afterwards, the notes are checked and cleaned for visual clarity. The data is returned as a dataframe, this way the user can see what the about page will look like for themselves. 

***

## **Setup**

***

In [4]:
import pandas as pd
import os
import yaml
from datetime import date
import numpy as np
from tqdm import tqdm

In [5]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths

# SharePoint
path_about   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents', 'Data', 'About Indicators.xlsx')

# Git
path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')

# Where this file lies. Saved this so that we can send test exports to this location
path_self = os.path.join(path_git, 'Python Code', 'About Tab Automation')

# yaml pathway
path_yaml = os.path.join(path_self, 'dict_about.yaml')

<>:8: SyntaxWarning: invalid escape sequence '\R'
<>:8: SyntaxWarning: invalid escape sequence '\R'
C:\Users\jchoy\AppData\Local\Temp\ipykernel_20564\4242418775.py:8: SyntaxWarning: invalid escape sequence '\R'
  path_about   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents', 'Data', 'About Indicators.xlsx')


In [4]:
# Reading in About_Indicators.xlsx

about_indicators = pd.ExcelFile(path_about)

# Empty dict
dict_about = {}

# Skip first sheet
for sheet_name in about_indicators.sheet_names[1:]:
    # make a df for each sheet
    df = pd.read_excel(about_indicators, sheet_name=sheet_name, header = None)

    # Initialize to hold info
    info_dict = {}
    notes = []  # This is done bc notes go into multiple rows sometimes. 

    # Processing each row in df
    for index, row in df.iterrows():
        key = row.iloc[0]  
        value = row.iloc[1]  

        # Checking if a row is a note (if key is NULL and value is present)
        if pd.isna(key) and not pd.isna(value):
            notes.append(value)
        else:
            if not isinstance(value, list):
                value = [value]

            if pd.notna(key) and key == 'Indicator Title':
                info_dict[key] = value 

            # Adding collected notes to dict
            if notes:
                info_dict['Notes'] = ' '.join(notes) # combines as a single string
                notes = [] 

            # adding pair in if valid
            if pd.notna(key):
                info_dict[key] = value

    # Joining the notes, would be nice to keep the formatting but need to do this
    if notes:
        info_dict['Notes'] = ' '.join(notes)

    # Use the sheet name directly as the key in the dictionary
    dict_about[sheet_name] = info_dict

# Probs won't need this code again, but just wanted to leave to document

In [6]:
# Convert the dict to a YAML string
yaml_string = yaml.dump(dict_about, sort_keys=False, default_flow_style=False)

# Save the YAML string to a file
with open('dict_about.yaml', 'w') as yaml_file:
    yaml.dump(dict_about, yaml_file, sort_keys=False, default_flow_style=False)

# All edits to YAML file are now done within the YAML file

Pop_1:
  Indicator Title:
  - Population
  Source:
  - California Department of Finance E-5 (current decade) and E-8 (prior) series
  Year(s):
  - 2000-2024
  Description:
  - Estimated population
  Geography:
  - County
  Website:
  - https://dof.ca.gov/forecasting/demographics/estimates/
  Last Updated:
  - 2024-04-01 00:00:00
  Notes: The Department of Finance prepares its population estimates in ten-year series.
    Each ten-year series begins with the results of the decennial census (census benchmark).
    Users should pay careful attention when using data that spans two decennial census
    periods, as the difference between the final year of the prior series and the
    first of the new may stem largely from the new census benchmark. The monitoring
    program uses the decennial census benchmark for the decennial years (2000, 2010,
    2020 etc.). The decennial census benchmark is for April 1st of the decennial year;
    all other population estimates are for January 1st of the 

***

## **Function/Script Building**

***

In [89]:
path_yaml = os.path.join(path_self, 'dict_about.yaml')

try:
    with open(path_yaml, 'r') as yaml_file:
        dict_about = yaml.load(yaml_file, Loader=yaml.SafeLoader)
    print(dict_about)
except FileNotFoundError:
    print(f"Error: The file at {path_yaml} does not exist.")
except Exception as e:
    print(f"An error occurred: {e}")

df_dicto = pd.DataFrame.from_dict(dict_about['ACS']['Pop_3']).T.reset_index().rename(columns = {'index': 'Metadata', 0: 'Description'})


{'DOF': {'Pop_1': {'Indicator Title': ['Population'], 'Source': ['California Department of Finance E-5 (current decade) and E-8 (prior) series'], 'Year(s)': ["str(np.min(df['Year'].unique())) + '-' + str(np.max(df['Year'].unique()))"], 'Description': ['Estimated population'], 'Geography': ['County'], 'Website': ['https://dof.ca.gov/forecasting/demographics/estimates/'], 'Last Updated': ["date.today().strftime('%Y-%m-%d')"], 'Notes': 'The Department of Finance prepares its population estimates in ten-year series. Each ten-year series begins with the results of the decennial census (census benchmark). Users should pay careful attention when using data that spans two decennial census periods, as the difference between the final year of the prior series and the first of the new may stem largely from the new census benchmark. The monitoring program uses the decennial census benchmark for the decennial years (2000, 2010, 2020 etc.). The decennial census benchmark is for April 1st of the dece

In [90]:
df_dicto[df_dicto['Metadata'] == 'Notes']['Description'].values[0]

'Each of these geographies has the total population estimate by race/ethnicity, as well as the percentage. \\n Key terms \\n Rolling Average: The data are reported in 5 year rolling averages. For example, the 2022 output is not the individual year of 2022, but the 5 year average of 2018-2022. Users looking for single year results should use the separate 1 year ACS series. \\n Margin of Error: all ACS outputs are estimates. The downloadable data includes the margin of error statistics. Users should pay particular to the margin of error, especially for smaller geographies and race/ethnicity groups with relatively less population.\\n State and County FIPS are census numeric codes for different geographies'

In [183]:
# Function / Script to write about pages. 

# Would probs be better to just make it a script, but func for now

# path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out) # need to define these

def write_about(sample_type, indicator_name, year_frame, name_output_xlsx, path_out_xlsx):

    import numpy as np
    import re
    import pandas as pd
    import os
    from datetime import date

    # Define path_self, and path_git here. These paths are still subject to change. 

    path_yaml = os.path.join(path_self, 'dict_about.yaml')
    
    try:
        with open(path_yaml, 'r') as yaml_file:
            dict_about = yaml.load(yaml_file, Loader=yaml.SafeLoader)
    except FileNotFoundError:
        print(f"Error: The file at {path_yaml} does not exist.")
    except Exception as e:
        print(f"An error occurred: {e}")

    df_dicto = pd.DataFrame.from_dict(dict_about[sample_type][indicator_name]).T.reset_index().rename(columns = {'index': 'Metadata', 0: 'Description'})

    def evaluate_expression(expr, context):
        try:
            # check each cell for code. Provide context for execution of code
            return eval(expr, {"np": np, "date": date, **context}) # "geography": geography
        except Exception as e:
            print(f"Error: Could not evaluate expression: '{expr}': {e}")
            return expr  # Returns the string if unable to be passed

    # Context for the dynamic variables
    context = {
        "np": np,
        "date": date,
        "df": year_frame  # Do this so that we can pass different frames.
        #, "geography": geography and we'd have to add geography to the evaluate_expression function. Nothing too crazy tho.
    }

    # Apply the function to the about tab. 
    df_dicto['Description'] = df_dicto['Description'].apply(
        lambda x: evaluate_expression(x, context) if isinstance(x, str) and ('np.' in x or 'date.' in x or 'df' in x) else x
    )
    
    # Split notes into rows, for visual clarity in about
    def split_notes(df):
        # Find the row with 'Notes', then use that to take the information
        notes_row = df[df['Metadata'] == 'Notes'].copy()
        notes = notes_row['Description'].values[0]
        
        # Separate based off of NewLines, make the rows with this
        lines = notes.split('\\n')

        # Make a blank row past the first one. This way, we don't have to see notes as a cell like 7 times.
        new_rows = [{'Metadata': 'Notes' if i == 0 else '', 'Description': line} for i, line in enumerate(lines) if line]
        
        # Create a df from the new separated rows. Drop the old notes row
        new_df = pd.DataFrame(new_rows)
        df_filtered = df[df['Metadata'] != 'Notes']
        
        # Combine original with new rows
        notes_df = pd.concat([df_filtered, new_df], ignore_index=True)
        
        return notes_df
    
    # Finally, we split the notes
    df_dicto = split_notes(df_dicto)

    # Display for the user
    print("Visual representation of the output for:", indicator_name)
    display(df_dicto)

    # Might be better to just return the dataframe. That way, the user can see what the output looks like.
    # Additionally, it could be nice to have it so that if the user needs to edit part of the about themselves, they can do so by the following:
    # df_dicto[df_dicto['Metadata'] == 'Row Val']['Description'].values[0] == 'CHANGE'
    # a little iffy on making Geography a dynamic variable, a lot of the workbooks are very descriptive with this field
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
        df_dicto.to_excel(writer, index = False, sheet_name = 'About')
    

In [27]:
# Function to write about page for each indicator
def write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0, estimate = None):
    '''
    User defined function to create/export about documentation for each indicator
    Inputs: .yaml file, specific indicator inputs (geography, sample type, ...), data frame to export, file paths, ...
    Uses user defined inputs to organize .yaml file subset into pandas data frame then exports to excel file sheet
    '''

    # Reads in .yaml file
    # Creates context object to update reactive objects defined in the .yaml file

    path_yaml = os.path.join(path_config0, 'dict_about.yaml')

    try:
        with open(path_yaml, 'r') as yaml_file:
            dict_about = yaml.load(yaml_file, Loader=yaml.SafeLoader)
    except FileNotFoundError:
        print(f"Error: The file at {path_yaml} does not exist.")
    except Exception as e:
        print(f"An error occurred: {e}")

    try:
        if estimate == '5YEAR':
            df_dicto = pd.DataFrame.from_dict(dict_about[estimate][sample_type][indicator_name]).T.reset_index().rename(columns={'index': 'Metadata', 0: 'Description'})
        elif estimate == '1YEAR':
            df_dicto = pd.DataFrame.from_dict(dict_about[estimate][sample_type][indicator_name]).T.reset_index().rename(columns={'index': 'Metadata', 0: 'Description'})
        else:
            df_dicto = pd.DataFrame.from_dict(dict_about[sample_type][indicator_name]).T.reset_index().rename(columns={'index': 'Metadata', 0: 'Description'})
    
    except KeyError as e:
        missing_key = e.args[0]
        print(f"Error: Sample '{missing_key}' not found in the nested structure.")
        raise

    df_dicto.loc[df_dicto['Metadata'] == 'Last Updated', 'Description'] = date.today().strftime('%Y-%m-%d')
    df_dicto.loc[df_dicto['Metadata'] == 'Year(s)'     , 'Description'] = f"{year_start}-{year_end}"
    df_dicto.loc[df_dicto['Metadata'] == 'Geography'   , 'Description'] = geography


    # Split notes into rows, for visual clarity in about
    # Find the row with 'Notes', then use that to take the information
    # Separate based off of NewLines, make the rows with this
    # Make a blank row past the first one. This way, we don't have to see notes as a cell like 7 times.
    # Create a df from the new separated rows. Drop the old notes row
    # Combine original with new rows
    # Finally, we split the notes
    def split_notes(df):
        notes_row = df[df['Metadata'] == 'Notes'].copy()
        notes = notes_row['Description'].values[0]

        lines = notes.split('\\n')
        new_rows = [{'Metadata': 'Notes' if i == 0 else '', 'Description': line} for i, line in enumerate(lines) if line]

        new_df = pd.DataFrame(new_rows)
        df_filtered = df[df['Metadata'] != 'Notes']

        notes_df = pd.concat([df_filtered, new_df], ignore_index=True)
        return notes_df
    
    df_dicto = split_notes(df_dicto)

    return df_dicto

***

## **Testing the Export**

***

In [9]:
# For this one, I tested on Pop_3

path_pop = "C:\\Users\\jchoy\\Sacramento Area Council of Governments\\Regional Monitoring and Reporting - Documents\\Data\\Vibrant and Inclusive Places\\People and Community\\Pop and Demographics\\Pop_3 Race\\Pop_3 MPO ACS1.xlsx"

In [18]:
# Both sample_type and indicator_name are variables we reference a lot in other workbooks
sample_type = 'PUMS'

indicator_name = 'Accessibility_4'

geography = 'MPOs'

year_start = '2011'

year_end = '2020'

# Might want to get rid of headers: metadata and information on export. 
estimate_test = write_about(sample_type, indicator_name, geography, year_start, year_end, path_self, estimate = '1YEAR')

# testing multiple nests
display(estimate_test)

,Metadata,Description
0,Indicator Title,Commute times by income bracket and by race/et...
1,Source,"Census PUMS, 1 year ACS"
2,Year(s),2011-2020
3,Description,Yearly estimate of head of household commute t...
4,Geography,MPOs
5,Website,https://www.census.gov/programs-surveys/acs/mi...
6,Last Updated,2024-09-09
7,Margin of Error Limit,5
8,Notes,CPI Inflation Adjustment Factors.xlsx (sharepo...
9,,"The income brackets (Low, Moderate, and High)..."


In [29]:
# Both sample_type and indicator_name are variables we reference a lot in other workbooks
sample_type = 'BLS'

indicator_name = 'Jobs_1'

geography = 'MPOs'

year_start = '2000'

year_end = '2004'

# Might want to get rid of headers: metadata and information on export. 
# tested this with estimate set to 1year and it didn't work! good
bls_test = write_about(sample_type, indicator_name, geography, year_start, year_end, path_self)

display(bls_test)

,Metadata,Description
0,Indicator Title,Total Jobs by MSA
1,Source,"BLS monthly "" State and Area Employment, Hours..."
2,Year(s),2000-2004
3,Description,Monthly totals of people employed on the MSA l...
4,Geography,MPOs
5,Website,https://www.bls.gov/sae/overview.htm
6,API Series,https://www.bls.gov/help/hlpforma.htm#SM
7,Last Updated,2024-09-10
8,Notes,The survey is updated on a monthly basis on th...
9,,Employment data refer to persons on establish...
